# 04 — Inference Server

For closed-loop evaluation against a simulator or a real robot,
loading the model in-process usually does not fit the latency /
memory budget. RLDX ships a small ZeroMQ inference server that
holds the model in GPU memory and answers `get_action` requests
over a socket.

This notebook launches the server as a subprocess, connects a
client, sends one request, and shuts the server down.

The canonical docs are in [`../docs/inference_server.md`](../docs/inference_server.md).


## 1. Start the server

`rldx/eval/run_rldx_server.py` is the entry point. The key flags:

| Flag | Purpose |
|---|---|
| `--model-path` | Checkpoint directory or HF repo id. |
| `--embodiment-tag` | `EmbodimentTag` value matching the client's observation format. |
| `--host` / `--port` | Where to bind the ZMQ socket. Default `127.0.0.1:5555`. |
| `--use-sim-policy-wrapper` | Wrap the policy in `RLDXSimPolicyWrapper` so it accepts flat `video.<key>` / `state.<key>` observations — use this when the client is an RLDX sim env. |
| `--denoising-timesteps` | Override the flow-matching denoising schedule (e.g. `0.0 0.1 0.3 0.6`). |
| `--deactivate-memory` | Load a memory-trained checkpoint as a memoryless model. |


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

REPO = Path.cwd().parent
MODEL = 'RLWRLD/RLDX-1-FT-ROBOCASA'
PORT = 20099

server_cmd = [
    'uv', 'run', 'python',
    str(REPO / 'rldx/eval/run_rldx_server.py'),
    '--model-path', MODEL,
    '--embodiment-tag', 'GENERAL_EMBODIMENT',
    '--host', '127.0.0.1',
    '--port', str(PORT),
    '--use-sim-policy-wrapper',
]
print('>>>', ' '.join(server_cmd))
# Uncomment to actually start. The server takes ~30-60s to load the
# model on the first run (checkpoint download + bf16 load).
# server = subprocess.Popen(server_cmd, cwd=REPO)
# time.sleep(60)  # TODO: poll the server health endpoint instead


## 2. Connect a client

`rldx.policy.server_client.PolicyClient` is the matching ZMQ
client. It inherits from `BasePolicy`, so its `get_action(obs)`
method takes the same flat observation dict the sim wrapper
accepts, serialises it with msgpack over the socket, and returns
a `(actions, info)` tuple. Pass `strict=False` on the client —
strict-mode validation only runs server-side.


In [ ]:
import numpy as np
from rldx.policy.server_client import PolicyClient

client = PolicyClient(host='127.0.0.1', port=PORT, strict=False)

B, Tv, Ts, H, W = 1, 4, 1, 256, 256
rng = np.random.default_rng(0)
obs = {
    'video.left_view':  rng.integers(0, 255, (B, Tv, H, W, 3), dtype=np.uint8),
    'video.right_view': rng.integers(0, 255, (B, Tv, H, W, 3), dtype=np.uint8),
    'video.wrist_view': rng.integers(0, 255, (B, Tv, H, W, 3), dtype=np.uint8),
    'state.end_effector_position_relative': np.zeros((B, Ts, 3), dtype=np.float32),
    'state.end_effector_rotation_relative': np.zeros((B, Ts, 4), dtype=np.float32),
    'state.gripper_qpos':                   np.zeros((B, Ts, 2), dtype=np.float32),
    'state.base_position':                  np.zeros((B, Ts, 3), dtype=np.float32),
    'state.base_rotation':                  np.zeros((B, Ts, 4), dtype=np.float32),
    'annotation.human.action.task_description': ('pick up the mug',),
}

# actions, info = client.get_action(obs)
# for k, v in actions.items():
#     print(f'  {k:32s} shape={v.shape} dtype={v.dtype}')


## 3. Sim policy wrapper — what it actually does

When you start the server with `--use-sim-policy-wrapper`, the
policy object in-process is an `RLDXSimPolicyWrapper`, not a bare
`RLDXPolicy`. The wrapper:

1. Accepts the flat `video.<key>` / `state.<key>` observation
   format that the RLDX-1 sim envs use, and validates shapes /
   dtypes against the model's modality config.
2. Converts flat keys to the nested `VLAStepData` format
   `RLDXPolicy` consumes internally.
3. Returns actions in the same flat key format.

If your client is a real robot speaking a different observation
schema, skip `--use-sim-policy-wrapper` and interact with the
`RLDXPolicy` API directly (see `rldx/policy/policy.py`).


## 4. Shut down

ZeroMQ sockets and the held GPU memory are released when the
server process exits. Always terminate it explicitly — orphaned
servers hold the GPU hostage for the next tenant.


In [ ]:
# if 'server' in globals():
#     server.terminate()
#     server.wait(timeout=30)
#     print('server exit code:', server.returncode)


## 5. Next

- See `rldx/eval/rollout_policy.py` for an end-to-end rollout
  driver that talks to this server and records success rate.
- See [`../docs/evaluation.md`](../docs/evaluation.md) for the
  canonical simulator-eval launcher.
